# SBERT Resume-Job Matching

This is the main scoring notebook. We use Sentence-BERT (all-MiniLM-L6-v2) to turn each resume and each job description into an embedding, and then we use cosine similarity as the matching score. A higher score means the resume looks more similar to the job description in vector space.

These scores are the input for the fairness analysis notebook.

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")
print("Files uploaded.")

In [ ]:
jobs = pd.read_csv("data/jobs.csv")
resumes = pd.read_csv("data/resume_variants.csv")
print("Jobs:", jobs.shape)
print("Resumes:", resumes.shape)

In [ ]:
def make_job_text(row):
    return f"{row['title']} {row['domain']} {row['company_name']} {row['job_description']}"

jobs["job_text"] = jobs.apply(make_job_text, axis=1)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT loaded.")

In [ ]:
# Encode jobs and resumes once, in batches, instead of inside a nested loop.
job_embeddings = model.encode(jobs["job_text"].tolist())
resume_embeddings = model.encode(resumes["resume_text"].tolist())
print("Shapes:", resume_embeddings.shape, job_embeddings.shape)

In [ ]:
rows = []
for i, resume_row in resumes.reset_index(drop=True).iterrows():
    for j, job_row in jobs.reset_index(drop=True).iterrows():
        score = float(cosine_similarity(
            resume_embeddings[i].reshape(1, -1),
            job_embeddings[j].reshape(1, -1),
        )[0][0])
        rows.append({
            "resume_id": resume_row["resume_id"],
            "version": resume_row["version"],
            "changed_signal": resume_row["changed_signal"],
            "job_id": job_row["job_id"],
            "job_title": job_row["title"],
            "similarity_score": score,
        })

scores_df = pd.DataFrame(rows)
print("Total scored pairs:", len(scores_df))
display(scores_df.head())

In [ ]:
scores_df.to_csv("results/sbert_scores.csv", index=False)
print("Saved results/sbert_scores.csv")

In [ ]:
from google.colab import files
files.download("results/sbert_scores.csv")